### Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

colors_palette = ['#ffa600','#ff6361','#bc5090','#58508d', '#65345a']
sex_palette = ['#58508d','#bc5090']
survival_palette = {0: "#e74c3c", 1: "#2ecc71"}

In [ ]:
import os
import random

random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = str(42)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Poszukiwanie hiperparametrów

## Titanic - klasyfikacja

Najpierw wczytujemy dane.

In [ ]:
data_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(data_url)

print(f"Dataset: {df.shape}")

df.head()

Dzielinmy zbiór danych na zbiór treninowy i testowy.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Następnie sprawdzamy jakie zmienne mamy i informacje o nich.

In [ ]:
print("Cols:")
print(train_df.columns.tolist())

print("\n Info train:")
print(train_df.info())

print("\n Info test:")
print(test_df.info())

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

Sprawdzmy czy kapitan utonął - należy wyszukać rekord z 'Capt.' w nazwie.



In [ ]:
captain_data = ...
print(captain_data[['Name', 'Age', 'Sex', 'Pclass', 'Survived']])

Sprawdzamy brakujące wartości.

In [ ]:
missing_count = ...
missing_percent = ...
missing_info = pd.concat([missing_count, missing_percent], axis=1)
missing_info.columns = ['Count', 'Percentage']
missing_info = missing_info[missing_info['Count'] > 0].sort_values(by='Percentage', ascending=False)
print("Missing values train:")
print(missing_info)

Zróbmy kilka wizualizacji danych - wykresy analizujące wpływ płci, klasy i wieku na przeżywalność.

In [ ]:
sns.set_theme(style="whitegrid")

gender_order = ['male', 'female']

# Przeżywalność wg płci i klasy
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.barplot(x='Pclass', y='Survived', hue='Sex', data=train_df, hue_order=gender_order, palette=sex_palette, ax=axes[0])
axes[0].set_title('Training Set: Survival by Sex and Class')

sns.barplot(x='Pclass', y='Survived', hue='Sex', data=test_df, hue_order=gender_order, palette=sex_palette, ax=axes[1])
axes[1].set_title('Test Set: Survival by Sex and Class')
plt.show()

In [ ]:
# Przeyżwalność wg wieku
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.histplot(... hue='Survived', kde=True, element="step", ax=axes[0], palette='magma')
axes[0].set_title('Training Set: Age vs Survival')

sns.histplot(... hue='Survived', kde=True, element="step", ax=axes[1], palette='magma')
axes[1].set_title('Test Set: Age vs Survival')
plt.show()

In [ ]:
# Rozkład pasażerów w klasach
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

sns.countplot(... data=train_df, ax=axes[0], palette=colors_palette[:3], legend=False)
axes[0].set_title('Training Set: Pclass Distribution')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylabel('Count')

sns.countplot(... data=test_df, ax=axes[1], palette=colors_palette[:3], legend=False)
axes[1].set_title('Test Set: Pclass Distribution')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylabel('Count')
plt.show()

In [ ]:
# Przeżywalność w zależności od klasy
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

sns.barplot(... hue='Pclass', data=train_df, ax=axes[0], palette=colors_palette[:3], legend=False)
axes[0].set_title('Training Set: Survival Rate by Pclass')
axes[0].set_ylabel('Survival Probability')

sns.barplot(... hue='Pclass', data=test_df, ax=axes[1], palette=colors_palette[:3], legend=False)
axes[1].set_title('Test Set: Survival Rate by Pclass')
axes[1].set_ylabel('Survival Probability')
plt.show()

Wypełnijmy braki i przesaklujmy dane.

Dane można uzupełnić lub usunąć te rekordy. Bezpieczniejsze zazwyczaj jest usunięcie.

W tym przypadku uzupełnij:
* wiek medianą,
* port (Embarked) modą.

Usuń całą kolumnę Cabin ponieważ posiada za dużo brakujących wartości oraz nieznaczące kolumny: Name, Ticket, PassengerId.

In [ ]:
train_median = ...
train_df['Age'] = ...

most_freq_port = ...
train_df['Embarked'] = ...

train_df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

Teraz należy wykonać to samo zdla zbioru testowego - zwróć uwagę czy użyjesz mediany/mody dla zbioru treningowego czy testowego.

In [ ]:
test_df['Age'] = ...
test_df['Embarked'] = ...

test_df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

Zakoduj zmienne kategoryczne w sposób zrozumiały dla komputera.

In [ ]:
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
test_df['Sex'] = ...

train_df = pd.get_dummies(train_df, columns=['Embarked'], prefix='Port', dtype=int)
test_df = ...

test_df = test_df.reindex(columns = train_df.columns, fill_value=0)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

Analiza korelacji

Sprawdzmy, które cechy miały największy wpływ na to, czy ktoś przeżył.

Macierz korelacji pokazuje matematyczną zależność między zmiennymi. Współczynnik bliski 1 oznacza silną zależność dodatnią, a bliski -1 silną zależność ujemną.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(..., annot=True, cmap='RdYlGn', fmt=".2f")
plt.title("Mapa korelacji cech na Titanicu")
plt.show()

Sprawdźmy jeszcze czy po podziale mamy równowagę klas - wykresem i wypisaniem procentowego udziału.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# Train Set
sns.countplot(... ax=axes[0], palette=survival_palette, legend=False)
axes[0].set_title('Training Set: Class Balance (Survived)')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['No (0)', 'Yes (1)'])
axes[0].set_ylabel('Count')

# Test Set
sns.countplot(... ax=axes[1], palette=survival_palette, legend=False)
axes[1].set_title('Test Set: Class Balance (Survived)')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No (0)', 'Yes (1)'])
axes[1].set_ylabel('Count')

plt.show()

train_balance = ... # %
test_balance = ... # %

print(f"Train Set - Survivors: {train_balance[1]:.2f}%, Non-survivors: {train_balance[0]:.2f}%")
print(f"Test Set  - Survivors: {test_balance[1]:.2f}%, Non-survivors: {test_balance[0]:.2f}%")

Podzielmy zbióry na X i y i przeskalujmy.

In [ ]:
X_train = ...
y_train = ...
X_test = ...
y_test = ...

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    ...,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    ...,
    columns=X_test.columns,
    index=X_test.index
)

Przygotujmy funkcję obliczającą metryki do oceny modeli - użyjmy gotowych z biblioteki sklearn.

In [ ]:
def calculate_metrics(y_train, y_test, y_pred_train, y_pred_test):

    metrics = {
        'Accuracy_test': accuracy_score(y_test, y_pred_test),
        'Recall_test': recall_score(y_test, y_pred_test),
        'Precision_test': precision_score(y_test, y_pred_test),
        'F1_test': f1_score(y_test, y_pred_test),

        'Accuracy_train': accuracy_score(y_train, y_pred_train),
        'Recall_train': recall_score(y_train, y_pred_train),
        'Precision_train': precision_score(y_train, y_pred_train),
        'F1_train': f1_score(y_train, y_pred_train)
    }
    return metrics

def add_model_results(results_df, model_name, metrics):
    new_row = pd.DataFrame([metrics], index=[model_name])
    return pd.concat([results_df, new_row])

results_table = pd.DataFrame()

Teraz kolejno przetestujemy różne modele -  nie bedziemy już używać domyślnych hiperparamterów. Do ich wyszukania użyjemy techniki GridSearch.

Zaczniemy od modelu bazowego - Regresji Logistycznej.

In [ ]:
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

lr_grid = GridSearchCV(LogisticRegression(max_iter=1000), lr_params, cv=cv, scoring='f1')
lr_grid.fit(X_train_scaled, y_train)

best_lr = lr_grid.best_estimator_
y_pred_test = best_lr.predict(X_test_scaled)
y_pred_train = best_lr.predict(X_train_scaled)
results_table = add_model_results(results_table, 'Logistic Regression', calculate_metrics(y_train, y_test, y_pred_train, y_pred_test))

print(f"Best LR Params: {lr_grid.best_params_}")
results_table

Teraz sprawdźmy SVM.

In [ ]:
svm_params = {
    ...
}

svm_grid = GridSearchCV(SVC(), svm_params, cv=cv, scoring='f1')
svm_grid.fit(...)

best_svm = svm_grid.best_estimator_
y_pred_test = ...
y_pred_train = ...
results_table = add_model_results(results_table, 'SVM', calculate_metrics(...))

print(f"Best SVM Params: {svm_grid.best_params_}")
results_table

Następnie Random Forest.

In [ ]:
rf_params = {
    ...
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=cv, scoring='f1')
rf_grid.fit(...)

best_rf = rf_grid.best_estimator_
y_pred_test = ...
y_pred_train = ...
results_table = add_model_results(results_table, 'Random Forest', calculate_metrics(...))

print(f"Best RF Params: {rf_grid.best_params_}")
results_table

Porównajmy z XGBoost.

In [ ]:
xgb_params = {
    ...
}

xgb_grid = GridSearchCV(XGBClassifier(), xgb_params, cv=cv, scoring='f1')
xgb_grid.fit(...)

best_xgb = xgb_grid.best_estimator_
y_pred_test = ...
y_pred_train = ...
results_table = add_model_results(results_table, 'XGBoost', calculate_metrics(...))

print(f"Best XGB Params: {xgb_grid.best_params_}")
results_table

I na końcu z LightGBM.

In [ ]:
lgbm_params = {
    ...
}

lgbm_grid = GridSearchCV(LGBMClassifier(verbose=-1), lgbm_params, cv=cv, scoring='f1')
lgbm_grid.fit(...)

best_lgbm = lgbm_grid.best_estimator_
y_pred_test = ...
y_pred_train = ...
results_table = add_model_results(results_table, 'LightGBM', calculate_metrics(...))

print(f"Best LightGBM Params: {lgbm_grid.best_params_}")
results_table

Dodajmy jeszcze model złożony - stacking.

Modele bazowe generują przewidywania, a następnie kolejny model (meta) uczy się, jak najlepiej połączyć te wyniki, by podjąć ostateczną decyzję. Użyj jako modeli bazowych użytych wyżej, poza Regresją Logistyczną, ten model neich będzie modelem meta.

In [ ]:
stacking_clf = StackingClassifier(
    estimators=[
        ('rf', best_rf),
        ('svm', best_svm),
        ('xgb', best_xgb),
        ('lgbm', best_lgbm)
    ],
    final_estimator=LogisticRegression(),
    cv=cv
)

stacking_clf.fit(X_train_scaled, y_train)
y_pred_test = ...
y_pred_train = ...

results_table = add_model_results(results_table, 'Ensemble: Stacking', calculate_metrics(...))

In [ ]:
results_table = results_table.sort_values(by='F1_test', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(
    x=results_table['F1_test'],
    y=results_table.index,
    hue=results_table.index,
    palette='magma',
    legend=False
)

plt.title('Final Model Comparison: F1 Score')
plt.xlabel('F1 Score')
plt.xlim(0.7, 0.9)
plt.show()

results_table

Sprawdź czy przeżyłbyś na titanicu.

In [ ]:
my_pclass =            # class: 1, 2 lub 3
my_sex =               # male 0; female 1
my_age =
my_sibsp =              # siblings / husband/wife on ship
my_parch =              # parents/children on ship
my_fare =            # ticket price (average 30)
port_c =
port_q =
port_s = 0

person_features = pd.DataFrame([[my_pclass, my_sex, my_age, my_sibsp, my_parch, my_fare, port_c, port_q, port_s]],
                                 columns=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Port_C', 'Port_Q', 'Port_S'])

prediction = ...predict(person_features)[0]
probability = ....predict_proba(person_features)[0][1]

print(f"Survival Probability: {probability:.2%}")

if prediction == 1:
    print("Result: YOU SURVIVED!")
else:
    print("Result: YOU DIED.")

####Bonus: Optymalizacja Bayesowska z biblioteką Optuna

GridSearch sprawdza wszystkie kombinacje na ślepo. Optuna wykorzystuje wnioskowanie bayesowskie, aby przewidzieć, które parametry mogą być dobre, bazując na poprzednich próbach. Dzięki temu przeszukuje przestrzeń parametrów znacznie szybciej i skuteczniej.

In [ ]:
!pip install -q optuna
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

Przetestujmy ją dla modelu XGBoost.
Mozesz testować różne zakresy parametrów i ustawień. Zacznij od  n_trials 30, zwiększaj to i sprawdź czy uda odszukać Ci się lepszy model.

In [ ]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5)
    }

    model = XGBClassifier(**param, random_state=42)
    score = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1').mean()
    return score

In [ ]:
n_trials = 30
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=42),
    direction="maximize",
)
study.optimize(objective, n_trials=n_trials)
print(f"Best Score (F1): {study.best_value:.4f}")
print("Best params:", study.best_params)

best_xgb_optuna = XGBClassifier(**study.best_params, random_state=42)
best_xgb_optuna.fit(X_train_scaled, y_train)
y_pred_test = best_xgb_optuna.predict(X_test_scaled)
y_pred_train = best_xgb_optuna.predict(X_train_scaled)

results_table = add_model_results(results_table, 'XGBoost (Optuna)', calculate_metrics(y_train, y_test, y_pred_train, y_pred_test))
results_table.sort_values(by='F1_test', ascending=False)

In [ ]:
plot_param_importances(study)

In [ ]:
plot_optimization_history(study)